In [ ]:
# RSNA Knee - FAST CPU preprocess (no header reads): list -> sample K by index ->
# decode in parallel -> 224 uint8 shards. Mono shard, so slice ORDER is irrelevant
# (MIL head pools with order-invariant attention) => skip the 819k-header-read bottleneck.
import os, glob, time, numpy as np, pandas as pd, pydicom, cv2
from concurrent.futures import ThreadPoolExecutor
K = 8; IMG = 224; SHARD_STUDIES = 30
base = None
for r in sorted(glob.glob('/kaggle/input/*')):
    h = glob.glob(r+'/**/train_series.csv', recursive=True)
    if h: base = os.path.dirname(h[0]); break
print('base', base)
ts = pd.read_csv(os.path.join(base,'train_series.csv'))
tr = pd.read_csv(os.path.join(base,'train.csv'))
PL = {'Sagittal':0,'Coronal':1,'Axial':2}
print('series', ts.shape, 'studies', tr.shape[0])


In [ ]:
def sampled_paths(sd, k):
    fs = sorted(glob.glob(sd+'/*.dcm'))       # filename sort = deterministic; order not critical (mono+MIL)
    n = len(fs)
    if n == 0: return []
    if n <= k: return fs
    lo, hi = int(n*.12), int(n*.88)
    if hi <= lo: lo, hi = 0, n
    return [fs[int(round(x))] for x in np.linspace(lo, hi-1, k)]
def slice_u8(p):
    try:
        d = pydicom.dcmread(p); a = d.pixel_array.astype(np.float32)
        a = a*float(getattr(d,'RescaleSlope',1) or 1)+float(getattr(d,'RescaleIntercept',0) or 0)
        lo,hi = np.percentile(a,0.5), np.percentile(a,99.5); a = np.clip((a-lo)/max(hi-lo,1e-6),0,1)
        return (cv2.resize(a,(IMG,IMG),interpolation=cv2.INTER_AREA)*255).astype(np.uint8)
    except Exception:
        return None
print('helpers ready')


In [ ]:
studies = list(tr['StudyInstanceUID'])
gold = list(tr[tr['ACL'].notna()]['StudyInstanceUID'])
order = gold + [s for s in studies if s not in set(gold)]
os.makedirs('shards', exist_ok=True)
buf_img, buf_meta, buf_ids = [], [], []
shard_n = 0; t0 = time.time()
ex = ThreadPoolExecutor(max_workers=32)
def flush():
    global shard_n, buf_img, buf_meta, buf_ids
    if not buf_img: return
    np.savez_compressed(f'shards/shard_{shard_n:02d}.npz', img=np.stack(buf_img),
                        meta=np.array(buf_meta, np.int32), study_ids=np.array(buf_ids))
    print(f'  saved shard_{shard_n:02d} imgs={len(buf_img)} {time.time()-t0:.0f}s', flush=True)
    shard_n += 1; buf_img, buf_meta, buf_ids = [], [], []

for si, study in enumerate(order):
    buf_ids.append(study)
    tasks = []   # (path, pid, fb)
    for _, sr in ts[ts['StudyInstanceUID']==study].iterrows():
        sd = os.path.join(base,'train_series',study,sr['SeriesInstanceUID'])
        if not os.path.isdir(sd): continue
        pid = PL.get(sr['Anatomical_Plane'],0); fb = int(sr['Fluid_Sensitive'])
        for p in sampled_paths(sd, K): tasks.append((p, pid, fb))
    if tasks:
        imgs = list(ex.map(lambda t: slice_u8(t[0]), tasks))
        for (p,pid,fb), img in zip(tasks, imgs):
            if img is not None: buf_img.append(img); buf_meta.append((si, pid, fb))
    if si % 200 == 0: print(f'{si}/{len(order)} imgs={len(buf_img)} {time.time()-t0:.0f}s', flush=True)
    if (si+1) % SHARD_STUDIES == 0: flush()
flush()
print('DONE shards', shard_n, f'{time.time()-t0:.0f}s',
      'MB', round(sum(os.path.getsize('shards/'+f) for f in os.listdir('shards'))/1e6,1))
